## XGBoost Classification with Optuna for Hyperparameter Optimization (HPO)
XGBoost offers a powerful tree-based boosting algorithm for classification. Yet, despite it's powerful capabilities, they are highly prone to overfitting and thus requires extensive hyperparameter search to ensure it's not overfitting. 

In this notebook, we use a single-node (8-core) to serially perform HPO search for a highly imbalanced classification dataset. We'll also leverage mlflow to log all the HPO search runs as a child run for organization. 

Compute specifications:

Note DBR 16.4ML Runtime with Photon and compute with 8 cores x 64GB memory. For more details see here:

```json
{
    "num_workers": 0,
    "cluster_name": "Single Node MLR",
    "spark_version": "16.4.x-cpu-ml-scala2.13",
    "spark_conf": {
        "spark.master": "local[*, 4]",
        "spark.databricks.cluster.profile": "singleNode"
    },
    "aws_attributes": {
        "first_on_demand": 1,
        "availability": "SPOT_WITH_FALLBACK",
        "zone_id": "auto",
        "spot_bid_price_percent": 100,
        "ebs_volume_count": 0
    },
    "node_type_id": "r6idn.2xlarge",
    "driver_node_type_id": "r6idn.2xlarge",
    "ssh_public_keys": [],
    "custom_tags": {
        "ResourceClass": "SingleNode"
    },
    "spark_env_vars": {},
    "autotermination_minutes": 60,
    "enable_elastic_disk": true,
    "init_scripts": [],
    "single_user_name": "jon.cheung@databricks.com",
    "enable_local_disk_encryption": false,
    "data_security_mode": "SINGLE_USER",
    "runtime_engine": "PHOTON",
}
```

In [0]:
%pip install xgboost optuna mlflow # we need at least mlflow > 3.0

dbutils.library.restartPython()

## Generate/load a dataset
Below, we're using sci-kit learn to generate a synthetic classification dataset with 1M rows, 50 features, and a class imbalance (98/02). 

In [0]:
from sklearn.datasets import make_classification

# Create an imbalanced dataset
# Here I'm generating 1M samples with 50 features and an imbalance of 98% of the classes being negative. 
X, y = make_classification(n_samples=1000000, 
                           n_features=50, 
                           n_informative=8, 
                           n_classes=2, 
                           class_sep=0.2,
                           weights=[0.98, 0.02])

## Create an XGBoost training function
F1-score is the best metric to use since we are working with an imbalanced classification dataset. We'll create a custom evaluation function and pass that to the XGboost training API. 

In [0]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score, recall_score
import numpy as np

# Create F1 score evaluation metric for XGBoost validation sets
def f1_eval(y_pred, dtrain):
    # We create a reverse F1 score since XGBoost aims to minimizes the function
    y_true = dtrain.get_label()
    err = 1-f1_score(y_true, np.round(y_pred))
    return 'reverse_f1_err', err
  
def train_xgboost(X, y, params=dict(), perform_validation:bool=True):
  
  if perform_validation:
    # This is specifically for hyperparameter search, where we spilt the data for evaluation.
    # Perform train test split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    # Create the design matrices for both train and validation sets
    train = xgb.DMatrix(X_train, y_train)
    validation = xgb.DMatrix(X_val, y_val)
    # Generate the evaluation datasets for validation 
    evaluation_sets = [(train, 'train'), (validation, 'validation')]
  else:  
    # this is for training the final model where we want to use all the data.
    train = xgb.DMatrix(X, y)  
    evaluation_sets = [(train, 'train')]
  
  # Train our model using the specified parameters and output the evaluation results
  evaluation_results = {}
  model = xgb.train(params, 
                    dtrain=train,
                    evals=evaluation_sets,
                    num_boost_round=params["num_boosting_round"],
                    early_stopping_rounds=params["early_stopping_rounds"],
                    custom_metric=f1_eval,
                    evals_result=evaluation_results)

  return model, evaluation_results

## (OPTIONAL) Train one model with our XGBoost training function

In [0]:
# Train one model
params = {"objective": "binary:logistic",
          "eta": .3,
          "num_boosting_round": 1000,
          "early_stopping_rounds": 10, 
          "nthread": 8
}
# mlflow.autolog(disable=True)
_, results = train_xgboost(X, y, params=params, perform_validation=True)

## Perform Bayesian hyperparameter sweep with Optuna and train our final model

The Optuna library is the new recommended hyperparameter optimization (HPO) for use in ML on Databricks, superseding Hyperopt. It's a well-maintained package with baked in parameter sampling and default Bayesian search for efficiency. 

In [0]:
import optuna
from functools import partial
import mlflow

# Create an objective function for optuna
def objective(parent_run_id:str , trial):
  
  # Create a child run within the parent run for each HPO search for organizing all the runs
  with mlflow.start_run(run_name=f'xgb_model_hpo_{str(trial._trial_id)}',
                        nested=True,
                        parent_run_id=parent_run_id) as child_run:
    # Define the parameter space to search
    params = {"objective": trial.suggest_categorical("objective", choices=["binary:logistic"]),
              "eta": trial.suggest_float("eta", .01, 1, log=True),
              "num_boosting_round": trial.suggest_int("num_boosting_round", 10, 1000), 
              "early_stopping_rounds": trial.suggest_int("early_stopping_rounds", 10, 100),
              "nthread": 16, 
    }
    
    # Train the model with the sampled hyperparameters
    _, evaluation_results = train_xgboost(X, 
                                          y,
                                          params=params,
                                           perform_validation=True
                                            )
    
    # Obtain the F1 score and log the metrics + params to mlflow
    val_f1_score = 1 - (evaluation_results['validation']['reverse_f1_err'][-1])
    train_f1_score = 1 - (evaluation_results['train']['reverse_f1_err'][-1])
    mlflow.log_metrics({"validation_f1": val_f1_score,
                        "training_f1": train_f1_score})
    mlflow.log_params(params)

  # output the validation f1 score as the objective to optimize
  return val_f1_score

In [0]:
# Set the parameters for the experiment (i.e. the number of HPO search trials and the experiment name)
# For my single-node 8-core machine, each trial takes about 2-3 minutes to run. My dataset is 1M rows x 50 features for a XGBoost classification model. Note that by default, XGBoost will use all the available threads (i.e. cores) to train each model.
n_trials = 2
experiment_name = '/Users/jon.cheung@databricks.com/xgboost-optuna'
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name='serial-search') as parent_run:
  study = optuna.create_study(direction="maximize")
  study.optimize(partial(objective, parent_run.info.run_id), n_trials=n_trials)

  best_trial = study.best_trial
  print(f"Best trial f1: {best_trial.value}")
  print("Best trial params: ")
  for key, value in best_trial.params.items():
      print(f"    {key}: {value}")
  
  # Train final model with the best parameters and all the data
  final_model, evaluation_results = train_xgboost(X, 
                                        y,
                                        params=best_trial.params,
                                          perform_validation=False
                                          )
  f1_score = 1 - (evaluation_results['train']['reverse_f1_err'][-1])
  mlflow.xgboost.log_model(final_model, "model")
  mlflow.log_metric("training_f1", f1_score)


In [0]:
best_trial.params

## Scaling to multi-node distributed HPO search
## 🚧🚧🚧🚧 UNDER CONSTRUCTION 🚧🚧🚧🚧

In [0]:
import mlflow
from mlflow.optuna.storage import MlflowStorage

# Set up an mlflow persistent storage across the cluster
experiment_name = '/Users/jon.cheung@databricks.com/xgboost-optuna'
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
mlflow_storage = MlflowStorage(experiment_id=experiment_id)

In [0]:
from mlflow.pyspark.optuna.study import MlflowSparkStudy

mlflow_study = MlflowSparkStudy(
    study_name="xgboost-distributed-tuning",
    storage=mlflow_storage,
)


with mlflow.start_run(run_name='parallel-search') as parent_run:
  mlflow_study._directions = ["maximize"]
  mlflow_study.optimize(partial(objective, parent_run.info.run_id), n_trials=16, n_jobs=8)

# n_jobs defines the number of parallel searches. Set this to the max number of autoscaling workers (assuming you want to use to use all the workers)

best_params = mlflow_study.best_params

In [0]:
best_params = mlflow_study.best_params